# Lecture 4 Recitation — Cross-Validation in Practice

This notebook is the hands-on companion to the lecture. We build the cross-validation loop explicitly, inspect fold scores, select alpha, and compare the selected model with a few fixed regularization strengths.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print('Training samples:', len(X_train))
print('Test samples:', len(X_test))


## Exercise 1 — Write the model factory

The regularization strength is the hyperparameter. Keep the hinge loss and L2 penalty fixed so that we isolate the effect of alpha.

In [ ]:
def make_model(alpha):
    return SGDClassifier(
        loss='hinge', penalty='l2', alpha=float(alpha),
        max_iter=5000, tol=1e-4, random_state=RANDOM_STATE
    )


## Exercise 2 — Implement one K-fold evaluation

For one alpha, return the individual fold accuracies and their mean. Printing every fold makes the validation process visible instead of hiding it inside a library call.

In [ ]:
def cv_score(X, y, alpha, n_splits=5):
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for fold, (train_idx, val_idx) in enumerate(splitter.split(X, y), 1):
        model = make_model(alpha)
        model.fit(X[train_idx], y[train_idx])
        score = accuracy_score(y[val_idx], model.predict(X[val_idx]))
        scores.append(score)
        print(f'fold {fold}: {score:.4f}')
    scores = np.asarray(scores)
    print(f'mean: {scores.mean():.4f}')
    return scores

example_scores = cv_score(X_train, y_train, alpha=0.05)


## Exercise 3 — Search for alpha*

Use a small candidate grid first. Then increase the grid if you want a finer search. Remember that the best alpha is selected using validation scores, not the test set.

In [ ]:
alphas = np.r_[1e-5, np.arange(0.01, 0.51, 0.02)]
means = []
stds = []

for alpha in alphas:
    scores = cv_score(X_train, y_train, alpha, n_splits=5)
    means.append(scores.mean())
    stds.append(scores.std())
    print('-' * 50)

means = np.asarray(means)
stds = np.asarray(stds)
best = int(np.argmax(means))
alpha_star = float(alphas[best])
print(f'alpha* = {alpha_star:.5g}')
print(f'mean CV accuracy = {means[best]:.4f}')


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(alphas, means, marker='o')
plt.fill_between(alphas, means - stds, means + stds, alpha=0.15)
plt.axvline(alpha_star, linestyle='--', label=f'alpha*={alpha_star:.5g}')
plt.xlabel('alpha')
plt.ylabel('mean validation accuracy')
plt.title('Hyperparameter selection by 5-fold cross-validation')
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


## Exercise 4 — Retrain and evaluate

Now retrain on all training samples with alpha*. Only now do we touch the test set.

In [ ]:
final_model = make_model(alpha_star)
final_model.fit(X_train, y_train)

train_pred = final_model.predict(X_train)
test_pred = final_model.predict(X_test)

print('Training accuracy:', accuracy_score(y_train, train_pred))
print('Test accuracy    :', accuracy_score(y_test, test_pred))


## Exercise 5 — Compare fixed alphas

This makes the role of hyperparameter selection concrete. Compare a weakly regularized model, the selected model, and a strongly regularized model on the same untouched test set.

Do not use these test results to change alpha; they are for final reporting only.

In [ ]:
comparison = [1e-5, alpha_star, 1.0]
for alpha in comparison:
    model = make_model(alpha)
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f'alpha={alpha:.5g} | train={train_acc:.4f} | test={test_acc:.4f}')


## Takeaway

Lecture 3 answered: **how do we optimize theta for a chosen objective?**

Lecture 4 answers: **how do we choose the hyperparameter that defines that objective?**

Cross-validation provides an estimate of generalization using only the training set. The final test set is reserved for the final, unbiased performance estimate.